In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src import skillcorner
from src.load import load_match, load_meta, list_matches

SC_MATCHES = [m for m in list_matches() if not m.startswith("metrica")]
RESTART_TAKER = {"throw_in_for", "goal_kick_for", "corner_for"}   # offside doesn't apply to these


def det(rows):
    """True only if there is exactly one row and its position was detected."""
    return len(rows) == 1 and bool(rows.is_detected.fillna(False).iloc[0])


def x_of(rows):
    """x position (attacking frame) of a single row, else NaN."""
    return rows.x_att.iloc[0] if len(rows) == 1 else np.nan


print(len(SC_MATCHES), "SkillCorner matches")

20 SkillCorner matches


In [2]:
out = []
for m in SC_MATCHES:
    home_id = load_meta(m)["teams"]["home"]["team_id"]
    ev = skillcorner.load_dynamic_events(m)
    # a pass = a player possession that ended in a pass; frame_end = the frame the ball is played
    passes = ev[(ev.event_type == "player_possession") & (ev.end_type == "pass")]
    df = load_match(m, columns=["frame_id", "object_type", "player_id", "team_id", "x_home_m", "is_detected"])
    frames = dict(tuple(df[df.frame_id.isin(passes.frame_end)].groupby("frame_id")))

    for p in passes.itertuples():
        f = frames[p.frame_end].copy()
        # attacking frame: the PASSER's team attacks towards +x
        f["x_att"] = f.x_home_m * (1 if str(p.team_id) == home_id else -1)

        pl = f[f.object_type == "player"]
        ball = f[f.object_type == "ball"]
        passer = pl[pl.player_id == str(p.player_id)]
        receiver = (pl[pl.player_id == str(int(p.player_targeted_id))]
                    if pd.notna(p.player_targeted_id) else pl.iloc[:0])

        # defenders = the other team; they defend the +x goal, so the deepest has the LARGEST x_att.
        # Ties broken by player_id so the result is the same on every machine.
        defs = pl[pl.team_id != str(p.team_id)].dropna(subset=["x_att"])
        defs = defs.sort_values(["x_att", "player_id"], ascending=[False, True], kind="stable")
        second_last = defs.iloc[1:2]

        out.append(dict(
            match=m, event_id=p.event_id, frame=p.frame_end, period=p.period,
            tracked=len(pl) > 0,
            passer=det(passer), receiver=det(receiver), ball=det(ball), second_last_def=det(second_last),
            restart=(p.start_type == "unknown") and (p.game_interruption_before in RESTART_TAKER),
            x_receiver=x_of(receiver), x_ball=x_of(ball), x_second_last=x_of(second_last),
        ))

res = pd.DataFrame(out)
res["is_usable"] = res[["passer", "receiver", "ball", "second_last_def"]].all(axis=1)
res["offside_possible"] = (~res.restart) & (res.x_receiver > 0) & (res.x_receiver > res.x_ball)
res["offside"] = res.offside_possible & (res.x_receiver > res.x_second_last)
print(f"passes checked: {len(res):,}")

passes checked: 16,072


In [3]:
print(f"all passes: {len(res):,}   usable: {res.is_usable.sum():,} ({res.is_usable.mean():.1%})")
display(res[["tracked", "passer", "receiver", "ball", "second_last_def", "is_usable"]].mean().map("{:.1%}".format))
display(res.groupby("match").is_usable.agg(total="size", usable="sum", share="mean").sort_values("share"))

all passes: 16,072   usable: 6,349 (39.5%)


tracked            100.0%
passer              96.9%
receiver            81.4%
ball                98.0%
second_last_def     45.2%
is_usable           39.5%
dtype: str

,total,usable,share
match,,,
2016236,823,205,0.249089
2010085,836,256,0.306220
2015213,736,231,0.313859
2007448,752,239,0.317819
2017461,654,214,0.327217
2011166,630,210,0.333333
1925299,912,309,0.338816
2013725,766,261,0.340731
2006363,845,289,0.342012


## 6. Offside-relevant passes

In [4]:
rel = res[res.offside_possible]
usable_rel = rel[rel.is_usable]
print(f"offside-possible:      {len(rel):,}   ({len(rel)/len(res):.1%} of all passes)")
print(f"  of which usable:     {len(usable_rel):,}   ({rel.is_usable.mean():.1%})")
print(f"  usable AND offside:  {usable_rel.offside.sum():,}   ({usable_rel.offside.mean():.1%} of usable)")
display(rel[["passer", "receiver", "ball", "second_last_def", "is_usable"]].mean().map("{:.1%}".format))

res.to_parquet(ROOT / "data" / "processed" / "qa" / "passes_skillcorner.parquet", index=False)

offside-possible:      5,279   (32.8% of all passes)
  of which usable:     3,120   (59.1%)
  usable AND offside:  85   (2.7% of usable)


passer             97.7%
receiver           82.3%
ball               98.6%
second_last_def    62.4%
is_usable          59.1%
dtype: str